# Transaction embeddings: CoLES, LATTE-style alignment, LLM4ES-style embeddings

This notebook is adapted for transaction/event-sequence data and contains three compact branches:

1. **CoLES** — close to the provided `pytorch-lifestream` example: preprocessing → random slices → GRU encoder → contrastive training → client embeddings.
2. **LATTE-style** — sequence embeddings are aligned with text-description embeddings using symmetric InfoNCE. In the original LATTE idea, compact behavioral statistics are verbalized, embedded by a frozen text encoder, and used as semantic supervision for the transaction encoder.
3. **LLM4ES-style** — transactions are serialized into text; optionally a small causal LM is fine-tuned with next-token prediction; user embeddings are extracted by mean-pooling hidden states.

The code is intentionally minimal. Start with CoLES, then run LATTE/LLM4ES as additional experimental branches.

## 0. Install dependencies

In Colab, run this cell once. If you are running locally, install the same packages in your environment.

In [ ]:
import sys

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    !{sys.executable} -m pip install -q pytorch-lifestream pytorch-lightning sentence-transformers transformers datasets accelerate scikit-learn

In [2]:
!pip install -q pytorch-lifestream pytorch-lightning sentence-transformers transformers datasets accelerate scikit-learn

## 1. Imports and configuration

Change only this section for your dataset.

Expected raw transaction table: one row = one transaction/event.

In [3]:
import os
import math
import random
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [14]:
# =========================
# EDIT THIS CONFIG
# =========================

# Path to your raw transaction data.
# Example: DATA_PATH = "/content/my_transactions.csv"
DATA_PATH = 'data (1)/transactions_train.csv'

# Optional target table for downstream evaluation.
# It must contain ID_COL and TARGET_COL. If target is already in DATA_PATH, set TARGET_PATH = None
#TARGET_PATH = None
#TARGET_COL = "target"

# Main sequence columns.
ID_COL = "client_id"       # e.g. wallet_id / user_id / address / client_id
TIME_COL = "event_time"    # e.g. timestamp / block_time / transaction_date

# Candidate features. The helper below keeps only columns that exist in your data.
CATEGORICAL_COLS = [
    "chain", "protocol", "tx_type", "asset", "asset_type", "direction",
    "merchant_category", "mcc", "small_group"
]

NUMERICAL_COLS = [
    "amount", "amount_usd", "asset_usd_value_change", "usd_value", "gas_fee_usd", "fee_usd"
]

# For fast first run, you can limit the number of users.
MAX_USERS = None  # e.g. 5000

# CoLES parameters.
MIN_SEQ_LEN = 10
SPLIT_COUNT = 5
SLICE_MIN = 10
SLICE_MAX = 200
BATCH_SIZE = 256
MAX_EPOCHS_COLES = 5
EMBEDDING_DIM = 256

In [15]:
transactions_train=pd.read_csv(DATA_PATH)

FileNotFoundError: [Errno 2] No such file or directory: 'data (1)/transactions_train.csv'

In [9]:
agg_features=transactions_train.groupby('client_id')['amount_rur'].agg(['sum','mean','std','min','max']).reset_index()

NameError: name 'transactions_train' is not defined

## 2. Load and lightly clean data

This cell intentionally does only minimal preprocessing: date normalization, feature selection, missing values, and optional user sampling.

In [ ]:
def read_table(path: str) -> pd.DataFrame:
    path = str(path)
    if path.endswith(".csv") or path.endswith(".csv.gz"):
        return pd.read_csv(path)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".xlsx"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")


def prepare_raw_transactions(
    df: pd.DataFrame,
    id_col: str,
    time_col: str,
    categorical_cols: list[str],
    numerical_cols: list[str],
    max_users: int | None = None,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    df = df.copy()

    if id_col not in df.columns:
        raise ValueError(f"ID_COL='{id_col}' is not in dataframe columns: {df.columns.tolist()[:20]}")
    if time_col not in df.columns:
        raise ValueError(f"TIME_COL='{time_col}' is not in dataframe columns: {df.columns.tolist()[:20]}")

    categorical_cols = [c for c in categorical_cols if c in df.columns]
    numerical_cols = [c for c in numerical_cols if c in df.columns]

    # Convert time to integer order. CoLES example often uses event_time_transformation='none'.
    if not np.issubdtype(df[time_col].dtype, np.number):
        dt = pd.to_datetime(df[time_col], errors="coerce", utc=True)
        if dt.notna().sum() == 0:
            # fallback: factorize arbitrary time strings
            df[time_col] = pd.factorize(df[time_col].astype(str))[0].astype("int64") + 1
        else:
            min_dt = dt.min()
            df[time_col] = ((dt - min_dt).dt.total_seconds() // 86400).fillna(0).astype("int64") + 1
    else:
        df[time_col] = pd.to_numeric(df[time_col], errors="coerce").fillna(0).astype("int64") + 1

    for c in categorical_cols:
        df[c] = df[c].astype("string").fillna("unknown")

    for c in numerical_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("float32")

    # Optional: useful for heavy-tailed transaction amounts.
    for c in numerical_cols:
        log_col = f"log_abs_{c}"
        df[log_col] = np.log1p(np.abs(df[c])).astype("float32")
    numerical_cols = numerical_cols + [f"log_abs_{c}" for c in numerical_cols]

    df = df.sort_values([id_col, time_col]).reset_index(drop=True)

    if max_users is not None:
        users = df[id_col].drop_duplicates().sample(min(max_users, df[id_col].nunique()), random_state=SEED)
        df = df[df[id_col].isin(users)].reset_index(drop=True)

    return df, categorical_cols, numerical_cols


transactions_train = read_table(DATA_PATH)
# trx_df, cat_cols, num_cols = prepare_raw_transactions(
#     raw_df, ID_COL, TIME_COL, CATEGORICAL_COLS, NUMERICAL_COLS, MAX_USERS
# )
# display(trx_df.head())
# print("users:", trx_df[ID_COL].nunique(), "transactions:", len(trx_df))
# print("categorical:", cat_cols)
# print("numerical:", num_cols)

## 3. Optional target loading for downstream tasks

Use this if you have labels such as churn, user segment, risk class, suspicious activity, etc.

In [ ]:
def load_target(raw_df: pd.DataFrame | None = None) -> pd.DataFrame | None:
    if TARGET_PATH is not None:
        target = read_table(TARGET_PATH)
    elif raw_df is not None and TARGET_COL in raw_df.columns:
        target = raw_df[[ID_COL, TARGET_COL]].drop_duplicates(ID_COL)
    else:
        return None

    target = target[[ID_COL, TARGET_COL]].dropna().drop_duplicates(ID_COL)
    return target

# target_df = load_target(raw_df)
# display(target_df.head() if target_df is not None else None)

# Branch A — CoLES embeddings

This branch follows the structure of the attached CoLES notebook:

`PandasDataPreprocessor → ColesDataset with SampleSlices → GRU sequence encoder → CoLESModule → inference embeddings`.

In [ ]:
from ptls.preprocessing import PandasDataPreprocessor
from ptls.nn import TrxEncoder, RnnSeqEncoder
from ptls.frames import PtlsDataModule
from ptls.frames.coles import CoLESModule, ColesDataset
from ptls.frames.coles.split_strategy import SampleSlices
from ptls.data_load.datasets import MemoryMapDataset, inference_data_loader
from ptls.data_load.iterable_processing import SeqLenFilter

In [ ]:
def build_ptls_dataset(
    trx_df: pd.DataFrame,
    id_col: str,
    time_col: str,
    cat_cols: list[str],
    num_cols: list[str],
):
    preprocessor = PandasDataPreprocessor(
        col_id=id_col,
        col_event_time=time_col,
        event_time_transformation="none",
        cols_category=cat_cols,
        cols_numerical=num_cols,
        return_records=True,
    )
    dataset = preprocessor.fit_transform(trx_df)
    dataset = sorted(dataset, key=lambda x: x[id_col])
    return dataset, preprocessor


def make_trx_encoder_params(preprocessor, trx_df, time_col, cat_cols, num_cols):
    cat_sizes = preprocessor.get_category_dictionary_sizes()

    embeddings = {}
    for c in cat_cols:
        size = int(cat_sizes.get(c, trx_df[c].nunique() + 1))
        embeddings[c] = {"in": size, "out": min(64, max(8, int(round(math.sqrt(size) * 2))))}

    # In the original CoLES example, event time can also be embedded when it is a small integer domain.
    max_time = int(trx_df[time_col].max())
    if max_time <= 100_000:
        embeddings[time_col] = {"in": max_time + 2, "out": 16}

    numeric_values = {c: "identity" for c in num_cols}

    return dict(
        embeddings_noise=0.003,
        numeric_values=numeric_values,
        embeddings=embeddings,
    )


def train_coles_embeddings(dataset, preprocessor, trx_df, cat_cols, num_cols):
    train_records, valid_records = train_test_split(dataset, test_size=0.2, random_state=SEED)

    trx_encoder_params = make_trx_encoder_params(preprocessor, trx_df, TIME_COL, cat_cols, num_cols)

    seq_encoder = RnnSeqEncoder(
        trx_encoder=TrxEncoder(**trx_encoder_params),
        hidden_size=EMBEDDING_DIM,
        type="gru",
    )

    model = CoLESModule(
        seq_encoder=seq_encoder,
        optimizer_partial=partial(torch.optim.Adam, lr=1e-3),
        lr_scheduler_partial=partial(torch.optim.lr_scheduler.StepLR, step_size=30, gamma=0.9),
    )

    train_dl = PtlsDataModule(
        train_data=ColesDataset(
            MemoryMapDataset(
                data=train_records,
                i_filters=[SeqLenFilter(min_seq_len=MIN_SEQ_LEN)],
            ),
            splitter=SampleSlices(
                split_count=SPLIT_COUNT,
                cnt_min=SLICE_MIN,
                cnt_max=SLICE_MAX,
            ),
        ),
        train_num_workers=0,
        train_batch_size=BATCH_SIZE,
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS_COLES,
        accelerator="cuda" if torch.cuda.is_available() else "cpu",
        devices=1,
        enable_progress_bar=True,
        logger=False,
    )

    trainer.fit(model, train_dl)
    return model, seq_encoder, trainer


def infer_coles_embeddings(model, records, id_col: str, batch_size: int = 256) -> pd.DataFrame:
    dl = inference_data_loader(records, num_workers=0, batch_size=batch_size)
    embeds = torch.vstack(model.trainer.predict(model, dl)).detach().cpu().numpy()
    out = pd.DataFrame(embeds, columns=[f"coles_{i}" for i in range(embeds.shape[1])])
    out[id_col] = [x[id_col] for x in records]
    return out

In [ ]:
# Run CoLES branch after loading trx_df.

# dataset, preprocessor = build_ptls_dataset(trx_df, ID_COL, TIME_COL, cat_cols, num_cols)
# coles_model, coles_seq_encoder, coles_trainer = train_coles_embeddings(dataset, preprocessor, trx_df, cat_cols, num_cols)
# coles_emb_df = infer_coles_embeddings(coles_model, dataset, ID_COL, batch_size=BATCH_SIZE)
# display(coles_emb_df.head())
# coles_emb_df.to_parquet("coles_embeddings.parquet", index=False)

# Branch B — LATTE-style transaction/text alignment

Compact version of LATTE:

1. Build statistical profile for each user from raw transactions.
2. Convert that profile into a short natural-language description.
3. Embed descriptions with a frozen sentence encoder.
4. Align CoLES sequence embeddings with text embeddings using symmetric InfoNCE.

This branch is useful when categorical transaction fields have semantic meaning, e.g. protocol, tx type, merchant category, asset, transaction direction.

In [ ]:
from sentence_transformers import SentenceTransformer
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
def top_values_text(df, col, k=5):
    if col not in df.columns:
        return ""
    vc = df[col].astype(str).value_counts().head(k)
    return ", ".join([f"{idx} ({cnt})" for idx, cnt in vc.items()])


def make_behavior_descriptions(
    trx_df: pd.DataFrame,
    id_col: str,
    time_col: str,
    cat_cols: list[str],
    num_cols_original: list[str],
    max_top_values: int = 5,
) -> pd.DataFrame:
    rows = []
    # Use original numerical columns, not generated log columns, for human-readable text.
    num_cols_original = [c for c in num_cols_original if c in trx_df.columns and not c.startswith("log_abs_")]

    for user_id, g in trx_df.groupby(id_col, sort=False):
        n_events = len(g)
        t_min, t_max = int(g[time_col].min()), int(g[time_col].max())
        active_span = max(1, t_max - t_min + 1)

        amount_parts = []
        for c in num_cols_original[:4]:
            amount_parts.append(
                f"{c}: total={g[c].sum():.2f}, mean={g[c].mean():.2f}, std={g[c].std(ddof=0):.2f}"
            )

        cat_parts = []
        for c in cat_cols[:6]:
            cat_parts.append(f"top {c}: {top_values_text(g, c, k=max_top_values)}")

        description = (
            f"User {user_id} has {n_events} transactions over {active_span} time units. "
            f"Activity intensity is {n_events / active_span:.3f} events per time unit. "
            f"Numerical behavior: {'; '.join(amount_parts) if amount_parts else 'no numerical fields'}. "
            f"Categorical behavior: {'; '.join(cat_parts) if cat_parts else 'no categorical fields'}."
        )
        rows.append({id_col: user_id, "description": description})

    return pd.DataFrame(rows)


def embed_text_descriptions(desc_df: pd.DataFrame, text_col: str = "description", model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    text_model = SentenceTransformer(model_name, device=DEVICE)
    text_emb = text_model.encode(
        desc_df[text_col].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    text_emb_df = pd.DataFrame(text_emb, columns=[f"text_{i}" for i in range(text_emb.shape[1])])
    text_emb_df[ID_COL] = desc_df[ID_COL].values
    return text_emb_df, text_model

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def symmetric_infonce(seq_z, text_z, temperature=0.07):
    seq_z = F.normalize(seq_z, dim=1)
    text_z = F.normalize(text_z, dim=1)
    logits = seq_z @ text_z.T / temperature
    labels = torch.arange(seq_z.size(0), device=seq_z.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def train_latte_alignment(
    coles_emb_df: pd.DataFrame,
    text_emb_df: pd.DataFrame,
    id_col: str,
    epochs: int = 10,
    batch_size: int = 256,
    lr: float = 1e-3,
):
    seq_cols = [c for c in coles_emb_df.columns if c.startswith("coles_")]
    text_cols = [c for c in text_emb_df.columns if c.startswith("text_")]

    pair_df = coles_emb_df[[id_col] + seq_cols].merge(text_emb_df[[id_col] + text_cols], on=id_col, how="inner")
    X_seq = torch.tensor(pair_df[seq_cols].values, dtype=torch.float32)
    X_text = torch.tensor(pair_df[text_cols].values, dtype=torch.float32)

    dl = DataLoader(TensorDataset(X_seq, X_text), batch_size=batch_size, shuffle=True, drop_last=False)

    head = ProjectionHead(in_dim=X_seq.shape[1], out_dim=X_text.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        head.train()
        losses = []
        for xb_seq, xb_text in dl:
            xb_seq = xb_seq.to(DEVICE)
            xb_text = xb_text.to(DEVICE)
            pred = head(xb_seq)
            loss = symmetric_infonce(pred, xb_text)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f"epoch={epoch+1:02d} loss={np.mean(losses):.4f}")

    head.eval()
    with torch.no_grad():
        aligned = head(X_seq.to(DEVICE)).cpu().numpy()
        aligned = aligned / np.maximum(np.linalg.norm(aligned, axis=1, keepdims=True), 1e-12)

    aligned_df = pd.DataFrame(aligned, columns=[f"latte_{i}" for i in range(aligned.shape[1])])
    aligned_df[id_col] = pair_df[id_col].values
    return aligned_df, head, pair_df

In [ ]:
# Run LATTE-style branch after CoLES branch.

# original_num_cols = [c for c in NUMERICAL_COLS if c in trx_df.columns]
# desc_df = make_behavior_descriptions(trx_df, ID_COL, TIME_COL, cat_cols, original_num_cols)
# display(desc_df.head())

# text_emb_df, sentence_model = embed_text_descriptions(desc_df)
# latte_emb_df, latte_head, latte_pairs = train_latte_alignment(coles_emb_df, text_emb_df, ID_COL, epochs=10)
# display(latte_emb_df.head())
# latte_emb_df.to_parquet("latte_style_embeddings.parquet", index=False)

# Branch C — LLM4ES-style embeddings

Compact version of LLM4ES:

1. Serialize each user's event sequence into text.
2. Optionally fine-tune a small causal LM with next-token prediction.
3. Extract user embeddings by mean-pooling hidden states from the last hidden layers.

For a first experiment, keep `DO_LLM_FINETUNE = False` and use frozen LM embeddings. Then switch it to `True` for a small fine-tuning run.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
LLM_MODEL_NAME = "distilgpt2"   # small and easy to run. For better quality, try a stronger causal LM.
MAX_EVENTS_PER_USER_TEXT = 80
MAX_TEXT_TOKENS = 512
DO_LLM_FINETUNE = False
LLM_FINETUNE_EPOCHS = 1


def serialize_user_events(
    trx_df: pd.DataFrame,
    id_col: str,
    time_col: str,
    cat_cols: list[str],
    num_cols_original: list[str],
    max_events: int = 80,
) -> pd.DataFrame:
    rows = []
    num_cols_original = [c for c in num_cols_original if c in trx_df.columns and not c.startswith("log_abs_")]

    use_cols = [time_col] + cat_cols[:6] + num_cols_original[:4]
    use_cols = [c for c in use_cols if c in trx_df.columns]

    for user_id, g in trx_df.groupby(id_col, sort=False):
        g = g.sort_values(time_col).tail(max_events)
        event_texts = []
        for _, r in g.iterrows():
            parts = [f"time={r[time_col]}"]
            for c in cat_cols[:6]:
                if c in r:
                    parts.append(f"{c}={r[c]}")
            for c in num_cols_original[:4]:
                if c in r:
                    parts.append(f"{c}={float(r[c]):.4f}")
            event_texts.append("(" + ", ".join(parts) + ")")

        text = (
            f"This is a chronological transaction history for user {user_id}. "
            f"Each event contains time, categorical attributes and numerical values. "
            f"Events: " + " ".join(event_texts)
        )
        rows.append({id_col: user_id, "llm_text": text})

    return pd.DataFrame(rows)


def load_causal_lm(model_name: str = LLM_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)
    return tokenizer, model

In [ ]:
def finetune_causal_lm_on_event_texts(text_df: pd.DataFrame, tokenizer, model):
    ds = Dataset.from_pandas(text_df[["llm_text"]].rename(columns={"llm_text": "text"}))

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_TEXT_TOKENS)

    tokenized = ds.map(tok, batched=True, remove_columns=["text"])
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir="llm4es_tmp",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=LLM_FINETUNE_EPOCHS,
        learning_rate=5e-5,
        logging_steps=20,
        save_strategy="no",
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized,
        data_collator=collator,
    )
    trainer.train()
    return model


@torch.no_grad()
def extract_llm4es_embeddings(text_df: pd.DataFrame, tokenizer, model, batch_size: int = 8, last_k_layers: int = 4):
    model.eval()
    all_embs = []
    texts = text_df["llm_text"].tolist()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_TOKENS,
            return_tensors="pt",
        ).to(DEVICE)

        out = model(**enc, output_hidden_states=True)
        hidden = torch.stack(out.hidden_states[-last_k_layers:], dim=0).mean(dim=0)  # [B, T, H]
        mask = enc["attention_mask"].unsqueeze(-1)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        pooled = F.normalize(pooled, dim=1)
        all_embs.append(pooled.cpu().numpy())

    embs = np.vstack(all_embs)
    emb_df = pd.DataFrame(embs, columns=[f"llm4es_{i}" for i in range(embs.shape[1])])
    emb_df[ID_COL] = text_df[ID_COL].values
    return emb_df

In [ ]:
# Run LLM4ES-style branch.

# original_num_cols = [c for c in NUMERICAL_COLS if c in trx_df.columns]
# llm_text_df = serialize_user_events(trx_df, ID_COL, TIME_COL, cat_cols, original_num_cols, max_events=MAX_EVENTS_PER_USER_TEXT)
# display(llm_text_df.head())

# tokenizer, causal_lm = load_causal_lm(LLM_MODEL_NAME)
# if DO_LLM_FINETUNE:
#     causal_lm = finetune_causal_lm_on_event_texts(llm_text_df, tokenizer, causal_lm)

# llm4es_emb_df = extract_llm4es_embeddings(llm_text_df, tokenizer, causal_lm, batch_size=8, last_k_layers=4)
# display(llm4es_emb_df.head())
# llm4es_emb_df.to_parquet("llm4es_style_embeddings.parquet", index=False)

# Downstream evaluation

Use the same downstream split for all embedding types. For classification, the helper reports accuracy and ROC-AUC when applicable. For regression, it reports RMSE and R².

In [ ]:
def evaluate_embeddings(
    emb_df: pd.DataFrame,
    target_df: pd.DataFrame,
    id_col: str,
    target_col: str,
    task: str = "classification",
    test_size: float = 0.2,
):
    df = emb_df.merge(target_df[[id_col, target_col]], on=id_col, how="inner").dropna(subset=[target_col])
    emb_cols = [c for c in df.columns if c != id_col and c != target_col]

    X = df[emb_cols].values
    y = df[target_col].values

    stratify = y if task == "classification" and len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=SEED, stratify=stratify
    )

    if task == "classification":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        result = {"n_labeled": len(df), "accuracy": accuracy_score(y_test, pred)}

        if len(np.unique(y)) == 2:
            proba = clf.predict_proba(X_test)[:, 1]
            result["roc_auc"] = roc_auc_score(y_test, proba)
        else:
            proba = clf.predict_proba(X_test)
            try:
                result["roc_auc_ovr"] = roc_auc_score(y_test, proba, multi_class="ovr")
            except Exception:
                pass
        return result

    if task == "regression":
        reg = Ridge(alpha=1.0)
        reg.fit(X_train, y_train)
        pred = reg.predict(X_test)
        return {
            "n_labeled": len(df),
            "rmse": mean_squared_error(y_test, pred, squared=False),
            "r2": r2_score(y_test, pred),
        }

    raise ValueError("task must be 'classification' or 'regression'")

In [ ]:
# Example evaluation.

# target_df = load_target(raw_df)
# if target_df is not None:
#     print("CoLES:", evaluate_embeddings(coles_emb_df, target_df, ID_COL, TARGET_COL, task="classification"))
#     print("LATTE-style:", evaluate_embeddings(latte_emb_df, target_df, ID_COL, TARGET_COL, task="classification"))
#     print("LLM4ES-style:", evaluate_embeddings(llm4es_emb_df, target_df, ID_COL, TARGET_COL, task="classification"))

# Practical comparison plan

Suggested order:

1. **CoLES first**: it is the cleanest baseline and should be fastest after preprocessing.
2. **LATTE-style second**: checks whether semantic descriptions of behavior add useful signal over pure event-sequence structure.
3. **LLM4ES-style third**: first run frozen LM embeddings, then optional small fine-tuning. This branch is usually slower and more sensitive to text serialization quality.

Expected outputs:

- `coles_embeddings.parquet`
- `latte_style_embeddings.parquet`
- `llm4es_style_embeddings.parquet`

For your project report, compare these embeddings under the same downstream protocol: same train/test split, same classifier/regressor, same metric.